E4: TF-IDF experiment. They train word-level and character-level classifiers, validate them, compare their ensemble, create a precision/recall/F1 table, and save all artifacts to /kaggle/working/e4_tfidf_artifacts.

I have kept this focused: no transformer retraining, no test inference, and no extra experiments.

One limitation: your E1 artifact listing contains preds.npy, but not probs.npy. The code checks whether preds.npy contains three-class logits/probabilities. If it does, it includes E1 in the ensemble. If it contains only hard labels, it evaluates the TF-IDF ensemble independently rather than pretending hard labels are probabilities.

In [1]:
import os
import json
import random
import warnings
import joblib
import numpy as np
import pandas as pd

from scipy.special import softmax

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix
)

from IPython.display import display

warnings.filterwarnings("ignore")

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# -----------------------------
# Dataset paths
# -----------------------------
BASE_PATH = "/kaggle/input/datasets/suchok/base-train-data"

TRAIN_PATH = f"{BASE_PATH}/train_transformer_40k.csv"
VAL_PATH = f"{BASE_PATH}/val_fixed.csv"

E1_PATH = (
    "/kaggle/input/datasets/suchok/"
    "experiment1-artifacts/experiments/e1_chunked"
)

E1_PREDS_PATH = f"{E1_PATH}/preds.npy"

# -----------------------------
# Output directory
# -----------------------------
OUT_DIR = "/kaggle/working/e4_tfidf_artifacts"
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------
# Label mapping
# -----------------------------
LABELS = ["human", "ai", "fusion"]

LABEL2ID = {
    "human": 0,
    "ai": 1,
    "fusion": 2
}

ID2LABEL = {
    v: k for k, v in LABEL2ID.items()
}

print("Configuration ready.")
print("Output directory:", OUT_DIR)

Configuration ready.
Output directory: /kaggle/working/e4_tfidf_artifacts


Cell 2 — Load and validate the data
This uses train_pool.csv for fitting and val_fixed.csv for evaluation. Neither validation nor test data is used to fit the vectorizers or classifiers.

In [2]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

print("\nTrain columns:", train_df.columns.tolist())
print("Validation columns:", val_df.columns.tolist())

# Detect text column
TEXT_COL = "text"
LABEL_COL = "label"

assert TEXT_COL in train_df.columns, "Missing text column in train."
assert LABEL_COL in train_df.columns, "Missing label column in train."
assert TEXT_COL in val_df.columns, "Missing text column in validation."
assert LABEL_COL in val_df.columns, "Missing label column in validation."

# Normalize text
X_train = train_df[TEXT_COL].fillna("").astype(str).tolist()
X_val = val_df[TEXT_COL].fillna("").astype(str).tolist()

# Normalize labels
y_train_raw = train_df[LABEL_COL].astype(str).str.lower().str.strip()
y_val_raw = val_df[LABEL_COL].astype(str).str.lower().str.strip()

unknown_train = set(y_train_raw.unique()) - set(LABEL2ID)
unknown_val = set(y_val_raw.unique()) - set(LABEL2ID)

assert not unknown_train, f"Unknown train labels: {unknown_train}"
assert not unknown_val, f"Unknown validation labels: {unknown_val}"

y_train = y_train_raw.map(LABEL2ID).to_numpy(dtype=np.int64)
y_val = y_val_raw.map(LABEL2ID).to_numpy(dtype=np.int64)

print("\nTraining class distribution:")
print(pd.Series(y_train_raw).value_counts())

print("\nValidation class distribution:")
print(pd.Series(y_val_raw).value_counts())

# Check possible overlap by ID if IDs exist
if "id" in train_df.columns and "id" in val_df.columns:
    train_ids = set(train_df["id"].astype(str))
    val_ids = set(val_df["id"].astype(str))

    overlap = train_ids.intersection(val_ids)

    print("\nTrain/validation ID overlap:", len(overlap))
    assert len(overlap) == 0, "Train and validation IDs overlap!"

print("\nData loaded successfully.")

Train shape: (40000, 15)
Validation shape: (40853, 15)

Train columns: ['id', 'text', 'lang', 'domain', 'label', 'n_chars', 'n_words', '_nw', 'ttr', 'rate_em_dash', 'rate_semicolon', 'rate_exclaim', 'rate_question', 'rate_ellipsis', 'rate_comma']
Validation columns: ['id', 'text', 'lang', 'domain', 'label', 'n_chars', 'n_words', '_nw', 'ttr', 'rate_em_dash', 'rate_semicolon', 'rate_exclaim', 'rate_question', 'rate_ellipsis', 'rate_comma']

Training class distribution:
label
ai        17909
fusion    12290
human      9801
Name: count, dtype: int64

Validation class distribution:
label
ai        18291
fusion    12552
human     10010
Name: count, dtype: int64

Train/validation ID overlap: 0

Data loaded successfully.


Cell 3 — Train word-level TF-IDF + Logistic Regression
Configuration:
- Word unigrams and bigrams.
- No stopword removal.
- No stemming or lemmatization.
- No punctuation removal.
- Vectorizer fitted only on training data.
- Multinomial Logistic Regression with class weighting.

In [3]:
print("Fitting word-level TF-IDF...")

word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.995,
    sublinear_tf=True,
    max_features=200_000,
    dtype=np.float32
)

X_train_word = word_vectorizer.fit_transform(X_train)
X_val_word = word_vectorizer.transform(X_val)

print("Word train matrix:", X_train_word.shape)
print("Word validation matrix:", X_val_word.shape)

print("\nTraining word Logistic Regression...")

word_model = LogisticRegression(
    C=4.0,
    solver="saga",
    max_iter=60,
    tol=1e-3,
    class_weight="balanced",
    random_state=SEED,
    verbose=1
)

word_model.fit(X_train_word, y_train)

word_val_probs = word_model.predict_proba(X_val_word)

print("\nWord model training completed.")
print("Probability shape:", word_val_probs.shape)

assert word_val_probs.shape == (len(X_val), 3)

np.save(
    f"{OUT_DIR}/word_val_probs.npy",
    word_val_probs
)

joblib.dump(
    word_vectorizer,
    f"{OUT_DIR}/word_vectorizer.joblib",
    compress=3
)

joblib.dump(
    word_model,
    f"{OUT_DIR}/word_classifier.joblib",
    compress=3
)

print("Word model artifacts saved.")

Fitting word-level TF-IDF...
Word train matrix: (40000, 200000)
Word validation matrix: (40853, 200000)

Training word Logistic Regression...
convergence after 24 epochs took 18 seconds

Word model training completed.
Probability shape: (40853, 3)


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:   18.3s finished


Word model artifacts saved.


Cell 4 — Train character-level TF-IDF + Logistic Regression
Character n-grams can capture spelling, punctuation, spacing, word endings, and stylistic patterns that word-level features may miss.

In [4]:
print("Fitting character-level TF-IDF...")

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.995,
    sublinear_tf=True,
    max_features=250_000,
    dtype=np.float32
)

X_train_char = char_vectorizer.fit_transform(X_train)
X_val_char = char_vectorizer.transform(X_val)

print("Character train matrix:", X_train_char.shape)
print("Character validation matrix:", X_val_char.shape)

print("\nTraining character Logistic Regression...")

char_model = LogisticRegression(
    C=3.0,
    solver="saga",
    max_iter=60,
    tol=1e-3,
    class_weight="balanced",
    random_state=SEED,
    verbose=1
)

char_model.fit(X_train_char, y_train)

char_val_probs = char_model.predict_proba(X_val_char)

print("\nCharacter model training completed.")
print("Probability shape:", char_val_probs.shape)

assert char_val_probs.shape == (len(X_val), 3)

np.save(
    f"{OUT_DIR}/char_val_probs.npy",
    char_val_probs
)

joblib.dump(
    char_vectorizer,
    f"{OUT_DIR}/char_vectorizer.joblib",
    compress=3
)

joblib.dump(
    char_model,
    f"{OUT_DIR}/char_classifier.joblib",
    compress=3
)

print("Character model artifacts saved.")

Fitting character-level TF-IDF...
Character train matrix: (40000, 250000)
Character validation matrix: (40853, 250000)

Training character Logistic Regression...
convergence after 20 epochs took 139 seconds


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:  2.3min finished



Character model training completed.
Probability shape: (40853, 3)
Character model artifacts saved.


Cell 5 — Evaluate individual models and create the precision table
This creates a table with precision, recall, F1, and support for each class, as well as macro and weighted averages.

In [5]:
def evaluate_model(name, probs, y_true):
    preds = np.argmax(probs, axis=1)

    macro_f1 = f1_score(
        y_true,
        preds,
        average="macro"
    )

    print(f"\n{name}")
    print("=" * 60)
    print(f"Macro-F1: {macro_f1:.6f}")

    report = classification_report(
        y_true,
        preds,
        labels=[0, 1, 2],
        target_names=LABELS,
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report).T

    print("\nPrecision / Recall / F1:")
    display(report_df.round(4))

    print("\nConfusion matrix:")
    cm = confusion_matrix(y_true, preds, labels=[0, 1, 2])

    cm_df = pd.DataFrame(
        cm,
        index=[f"True_{x}" for x in LABELS],
        columns=[f"Pred_{x}" for x in LABELS]
    )

    display(cm_df)

    report_df.to_csv(
        f"{OUT_DIR}/{name.lower().replace(' ', '_')}_precision_table.csv"
    )

    pd.DataFrame(
        cm,
        index=LABELS,
        columns=LABELS
    ).to_csv(
        f"{OUT_DIR}/{name.lower().replace(' ', '_')}_confusion_matrix.csv"
    )

    return macro_f1, report_df, cm_df


word_f1, word_report, word_cm = evaluate_model(
    "Word TFIDF",
    word_val_probs,
    y_val
)

char_f1, char_report, char_cm = evaluate_model(
    "Character TFIDF",
    char_val_probs,
    y_val
)


Word TFIDF
Macro-F1: 0.839636

Precision / Recall / F1:


,precision,recall,f1-score,support
human,0.7220,0.8158,0.7660,10010.0000
ai,0.9753,0.9849,0.9801,18291.0000
fusion,0.8245,0.7272,0.7728,12552.0000
accuracy,0.8643,0.8643,0.8643,0.8643
macro avg,0.8406,0.8426,0.8396,40853.0000
weighted avg,0.8669,0.8643,0.8639,40853.0000



Confusion matrix:


,Pred_human,Pred_ai,Pred_fusion
True_human,8166,69,1775
True_ai,108,18015,168
True_fusion,3036,388,9128



Character TFIDF
Macro-F1: 0.865802

Precision / Recall / F1:


,precision,recall,f1-score,support
human,0.7588,0.8534,0.8033,10010.0000
ai,0.9807,0.9841,0.9824,18291.0000
fusion,0.8591,0.7692,0.8117,12552.0000
accuracy,0.8861,0.8861,0.8861,0.8861
macro avg,0.8662,0.8689,0.8658,40853.0000
weighted avg,0.8890,0.8861,0.8861,40853.0000



Confusion matrix:


,Pred_human,Pred_ai,Pred_fusion
True_human,8543,44,1423
True_ai,129,18001,161
True_fusion,2587,310,9655


Cell 6 — Try the word + character ensemble
Start with equal contributions and then search for the best word/character weight using the fixed validation set.

In [6]:
print("Searching for best word/character ensemble weights...")

best_score = -1
best_word_weight = None
best_char_weight = None
best_tfidf_probs = None

for word_weight in np.arange(0.0, 1.001, 0.025):
    char_weight = 1.0 - word_weight

    probs = (
        word_weight * word_val_probs
        + char_weight * char_val_probs
    )

    preds = np.argmax(probs, axis=1)

    score = f1_score(
        y_val,
        preds,
        average="macro"
    )

    if score > best_score:
        best_score = score
        best_word_weight = float(word_weight)
        best_char_weight = float(char_weight)
        best_tfidf_probs = probs.copy()

print("\nBest TF-IDF ensemble:")
print("Word weight:", best_word_weight)
print("Character weight:", best_char_weight)
print("Validation macro-F1:", best_score)

# Save ensemble probabilities
np.save(
    f"{OUT_DIR}/tfidf_ensemble_val_probs.npy",
    best_tfidf_probs
)

with open(f"{OUT_DIR}/tfidf_ensemble_config.json", "w") as f:
    json.dump(
        {
            "word_weight": best_word_weight,
            "character_weight": best_char_weight,
            "validation_macro_f1": float(best_score),
            "class_order": LABELS
        },
        f,
        indent=4
    )

Searching for best word/character ensemble weights...

Best TF-IDF ensemble:
Word weight: 0.15000000000000002
Character weight: 0.85
Validation macro-F1: 0.8668590129530332


Cell 7 — Check whether E1 validation predictions can be used
Your E1 artifact directory has preds.npy, but we need three-class probabilities for a proper probability ensemble.

This cell checks its shape and contents. If it contains three columns of logits or probabilities aligned with val_fixed.csv, it will be usable. If it contains only class IDs, it will not be used for probability blending.

In [7]:
e1_val_probs = None

if os.path.exists(E1_PREDS_PATH):
    e1_array = np.load(E1_PREDS_PATH)

    print("E1 preds.npy shape:", e1_array.shape)
    print("Validation row count:", len(y_val))

    if e1_array.ndim == 2 and e1_array.shape == (len(y_val), 3):
        # If already probabilities, keep them.
        # Otherwise, interpret the values as logits.
        row_sums = e1_array.sum(axis=1)

        is_probability = (
            np.all(e1_array >= 0)
            and np.all(e1_array <= 1)
            and np.allclose(row_sums, 1.0, atol=1e-3)
        )

        if is_probability:
            e1_val_probs = e1_array.astype(np.float64)
            print("E1 array appears to contain probabilities.")
        else:
            e1_val_probs = softmax(
                e1_array.astype(np.float64),
                axis=1
            )
            print("E1 array appears to contain logits; applied softmax.")

        print("E1 validation probabilities loaded.")

    else:
        print(
            "E1 predictions are not a three-class probability/logit matrix "
            "aligned with validation. E1 will not be included in this blend."
        )

else:
    print("E1 preds.npy not found. E1 will not be included.")

if e1_val_probs is not None:
    assert e1_val_probs.shape == (len(y_val), 3)

E1 preds.npy shape: (40853,)
Validation row count: 40853
E1 predictions are not a three-class probability/logit matrix aligned with validation. E1 will not be included in this blend.


Cell 8 — Final E4 ensemble and validation
If E1 probabilities are available, this cell searches for a constrained three-model ensemble. Otherwise, it uses the best word/character ensemble from Cell 6.

In [8]:
if e1_val_probs is not None:

    print("Searching E1 + word TF-IDF + character TF-IDF ensemble...")

    best_score = -1
    best_weights = None
    best_probs = None

    # E1 is the main model.
    # TF-IDF models provide complementary lexical signals.
    step = 0.025

    for word_weight in np.arange(0.0, 0.351, step):
        for char_weight in np.arange(0.0, 0.351, step):

            e1_weight = 1.0 - word_weight - char_weight

            if e1_weight < 0.50:
                continue

            probs = (
                e1_weight * e1_val_probs
                + word_weight * word_val_probs
                + char_weight * char_val_probs
            )

            preds = np.argmax(probs, axis=1)

            score = f1_score(
                y_val,
                preds,
                average="macro"
            )

            if score > best_score:
                best_score = score
                best_weights = {
                    "e1": float(e1_weight),
                    "word": float(word_weight),
                    "char": float(char_weight)
                }
                best_probs = probs.copy()

    final_val_probs = best_probs
    final_score = best_score

else:
    print("Using word + character TF-IDF ensemble.")

    final_val_probs = best_tfidf_probs
    final_score = best_score

    best_weights = {
        "word": best_word_weight,
        "char": best_char_weight
    }

print("\nFinal E4 validation result")
print("=" * 60)
print("Weights:", best_weights)
print("Macro-F1:", final_score)

np.save(
    f"{OUT_DIR}/e4_final_val_probs.npy",
    final_val_probs
)

with open(f"{OUT_DIR}/e4_final_config.json", "w") as f:
    json.dump(
        {
            "weights": best_weights,
            "validation_macro_f1": float(final_score),
            "class_order": LABELS,
            "includes_e1": e1_val_probs is not None
        },
        f,
        indent=4
    )

final_f1, final_report, final_cm = evaluate_model(
    "E4 Final Ensemble",
    final_val_probs,
    y_val
)

Using word + character TF-IDF ensemble.

Final E4 validation result
Weights: {'word': 0.15000000000000002, 'char': 0.85}
Macro-F1: 0.8668590129530332

E4 Final Ensemble
Macro-F1: 0.866859

Precision / Recall / F1:


,precision,recall,f1-score,support
human,0.7580,0.8590,0.8054,10010.0000
ai,0.9811,0.9853,0.9832,18291.0000
fusion,0.8635,0.7663,0.8120,12552.0000
accuracy,0.8871,0.8871,0.8871,0.8871
macro avg,0.8675,0.8702,0.8669,40853.0000
weighted avg,0.8903,0.8871,0.8870,40853.0000



Confusion matrix:


,Pred_human,Pred_ai,Pred_fusion
True_human,8599,41,1370
True_ai,118,18022,151
True_fusion,2627,306,9619


Cell 9 — Save the experiment summary and all remaining artifacts

In [9]:
summary = {
    "experiment": "E4_TFIDF_ensemble",
    "seed": SEED,
    "train_rows": len(X_train),
    "validation_rows": len(X_val),
    "classes": LABELS,

    "word_tfidf": {
        "ngram_range": [1, 2],
        "max_features": 200000,
        "min_df": 2,
        "max_df": 0.995,
        "C": 4.0,
        "max_iter": 60,
        "validation_macro_f1": float(word_f1)
    },

    "char_tfidf": {
        "ngram_range": [3, 5],
        "max_features": 250000,
        "min_df": 2,
        "max_df": 0.995,
        "C": 3.0,
        "max_iter": 60,
        "validation_macro_f1": float(char_f1)
    },

    "final_ensemble": {
        "weights": best_weights,
        "validation_macro_f1": float(final_f1),
        "includes_e1": e1_val_probs is not None
    }
}

with open(f"{OUT_DIR}/experiment_summary.json", "w") as f:
    json.dump(summary, f, indent=4)

# Save validation labels for future reproducible comparisons
np.save(
    f"{OUT_DIR}/validation_labels.npy",
    y_val
)

# Save validation IDs if available
if "id" in val_df.columns:
    np.save(
        f"{OUT_DIR}/validation_ids.npy",
        val_df["id"].to_numpy()
    )

print("All E4 artifacts saved.")
print("\nFiles:")
for filename in sorted(os.listdir(OUT_DIR)):
    filepath = os.path.join(OUT_DIR, filename)
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"{filename:45s} {size_mb:8.2f} MB")

All E4 artifacts saved.

Files:
char_classifier.joblib                            2.64 MB
char_val_probs.npy                                0.47 MB
char_vectorizer.joblib                            2.40 MB
character_tfidf_confusion_matrix.csv              0.00 MB
character_tfidf_precision_table.csv               0.00 MB
e4_final_config.json                              0.00 MB
e4_final_ensemble_confusion_matrix.csv            0.00 MB
e4_final_ensemble_precision_table.csv             0.00 MB
e4_final_val_probs.npy                            0.94 MB
experiment_summary.json                           0.00 MB
tfidf_ensemble_config.json                        0.00 MB
tfidf_ensemble_val_probs.npy                      0.94 MB
validation_ids.npy                                0.35 MB
validation_labels.npy                             0.31 MB
word_classifier.joblib                            2.14 MB
word_tfidf_confusion_matrix.csv                   0.00 MB
word_tfidf_precision_table.csv          

In [10]:
comparison = pd.DataFrame([
    {
        "Model": "Word TF-IDF",
        "Macro-F1": word_f1
    },
    {
        "Model": "Character TF-IDF",
        "Macro-F1": char_f1
    },
    {
        "Model": "Word + Character ensemble",
        "Macro-F1": best_score if e1_val_probs is None else np.nan
    },
    {
        "Model": "E4 final ensemble",
        "Macro-F1": final_f1
    }
])

print("E4 validation comparison:")
display(comparison.round(5))

comparison.to_csv(
    f"{OUT_DIR}/e4_model_comparison.csv",
    index=False
)

print("\nExperiment finished.")
print("Artifacts directory:", OUT_DIR)

E4 validation comparison:


,Model,Macro-F1
0,Word TF-IDF,0.83964
1,Character TF-IDF,0.86580
2,Word + Character ensemble,0.86686
3,E4 final ensemble,0.86686



Experiment finished.
Artifacts directory: /kaggle/working/e4_tfidf_artifacts
